# NB3 — Grover search on real IBM quantum hardware (E10)
Runs the same single-target Grover circuits as NB1 (n = 2–5, k = 0…2R_opt+2, ancilla-assisted MCX) on an IBM
quantum processor through the free **Open Plan** (up to 10 min QPU time per month), then compares the hardware
success probabilities with (i) Eq. (18) fitted to the hardware data, (ii) a noisy simulation of the *same*
transpiled circuits with the device's current calibration, and (iii) the calibration-only rate λ_cal.

**Budget:** 32 circuits per transpiler seed × 4096 shots in **one** job. Run 1 (one seed, ibm_fez, 2026-10-29) used 38 s of QPU time; run 2 with two seeds (`SEEDS_TRANSPILER = [0, 1]`, 64 circuits) should use about 80 s. Check the usage printed after the job.

**Run 2 (repeat for the paper):** set `RUN_TAG = "run2"`, keep `SEEDS_TRANSPILER = [0, 1]`, and run on a different day from run 1. If the Open Plan offers a second processor (e.g. one listed on your dashboard other than ibm_fez), run the notebook once more with `RUN_TAG = "run3"` and `BACKEND_NAME` set to it. Stay in the session until the job finishes (run 1 finished in a few minutes), because the transpiled circuits are saved at submission.

### Before you start (one time)
1. Create a free account at **quantum.cloud.ibm.com** and copy your **API key** (and, if shown, the instance **CRN**).
2. In Kaggle: *Add-ons → Secrets* → add `IBM_QUANTUM_TOKEN` (the API key) and, optionally, `IBM_QUANTUM_INSTANCE` (the CRN). Tick both secrets for this notebook. **Never paste the key into a cell.**
3. *Settings → Internet: On.*

### Three run modes (set `MODE` in the next cell)
| MODE | What it does | QPU time |
|---|---|---|
| `"test"` | Whole pipeline on the fake backend `FakeFez` (no account needed). Run this first. | none |
| `"submit"` | Transpiles for the least-busy real backend, submits one job, saves `job_id.json`, then waits for the result | ≈ 1–3 min |
| `"retrieve"` | Loads a job that was submitted earlier (queues can take hours) using `JOB_ID` and finishes the analysis | none |

If the Kaggle session ends while the job is queued, the job keeps running at IBM. Re-open the notebook, set `MODE = "retrieve"` and `JOB_ID = "..."` (printed at submission and saved in `job_id.json`), and run all cells.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "qiskit==2.6.1", "qiskit-aer==0.18.0", "qiskit-ibm-runtime==0.51.0"], check=False)

In [ ]:
import os, json, time, datetime, numpy as np, pandas as pd
MODE = "test"            # "test" | "submit" | "retrieve"
JOB_ID = ""              # only for MODE = "retrieve"
BACKEND_NAME = None      # None = least busy operational backend; or e.g. "ibm_fez"
RUN_TAG = "run2"         # output folder suffix: results_hw_run2
NS = [2, 3, 4, 5]
SHOTS = 4096
SEEDS_TRANSPILER = [0, 1]  # two layouts per circuit -> layout-to-layout variability on hardware
OUT = ("/kaggle/working/" if os.path.isdir("/kaggle/working") else "") + f"results_hw_{RUN_TAG}"
os.makedirs(OUT, exist_ok=True)
print("MODE =", MODE, "| output folder:", OUT)

In [ ]:
"""Shared code for the Grover-under-noise experiments (E1-E4).
Qiskit >= 2.0, qiskit-aer >= 0.17, qiskit-ibm-runtime >= 0.40 (fake backends)."""
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)  # QuantumCircuit.mcx(mode=...) deprecated in Qiskit 2.1; behaviour unchanged in 2.6.1
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile

# ---------- theory (Eqs. 10, 13, 18-20 of the manuscript) ----------
def theta(N, M=1):
    return np.arcsin(np.sqrt(M / N))

def p_ideal(k, N, M=1):
    return np.sin((2 * np.asarray(k) + 1) * theta(N, M)) ** 2

def p_noisy(k, N, lam, M=1):
    q = 1.0 - lam
    k = np.asarray(k, dtype=float)
    return q ** k * p_ideal(k, N, M) + (1 - q ** k) * M / N

def r_opt(N, M=1):
    return int(round(np.pi / (4 * theta(N, M)) - 0.5))

# ---------- circuits ----------
def _mcz(qc, qubits, anc, mode):
    """Multi-controlled Z on `qubits` (last one is the target of the MCX)."""
    *ctrl, tgt = qubits
    qc.h(tgt)
    if len(ctrl) == 1:
        qc.cx(ctrl[0], tgt)
    elif len(ctrl) == 2:
        qc.ccx(ctrl[0], ctrl[1], tgt)
    elif mode == "v-chain" and anc is not None and len(anc) > 0:
        qc.mcx(ctrl, tgt, ancilla_qubits=list(anc), mode="v-chain")
    else:
        qc.mcx(ctrl, tgt)  # synthesised by the transpiler (ancilla-free)
    qc.h(tgt)

def grover_circuit(n, target, k, mode="noancilla", measure=True):
    """n-qubit single-target Grover circuit with k iterations.
    Qubit i carries bit 2**i, so the measured bitstring equals format(target, f'0{n}b')
    and x1 (MSB in the manuscript convention) is qubit n-1."""
    q = QuantumRegister(n, "q")
    regs = [q]
    anc = None
    if mode == "v-chain" and n >= 4:
        anc = QuantumRegister(n - 3, "a")
        regs.append(anc)
    if measure:
        regs.append(ClassicalRegister(n, "c"))
    qc = QuantumCircuit(*regs, name=f"grover_n{n}_k{k}")
    zeros = [i for i in range(n) if not (target >> i) & 1]
    qc.h(q)
    for _ in range(k):
        # oracle O = I - 2|x0><x0|
        if zeros:
            qc.x([q[i] for i in zeros])
        _mcz(qc, list(q), anc, mode)
        if zeros:
            qc.x([q[i] for i in zeros])
        # diffusion D = 2|psi0><psi0| - I  (up to global phase -1)
        qc.h(q); qc.x(q)
        _mcz(qc, list(q), anc, mode)
        qc.x(q); qc.h(q)
    if measure:
        qc.measure(q, regs[-1])
    return qc

# ---------- resource metrics ----------
TWO_Q = ("ecr", "cz", "cx")

def resources(tqc):
    ops = tqc.count_ops()
    n2q = sum(v for g, v in ops.items() if g in TWO_Q)
    active = {tqc.find_bit(qb).index for inst in tqc.data
              if inst.operation.name not in ("barrier", "delay") for qb in inst.qubits}
    d2 = tqc.depth(filter_function=lambda inst: inst.operation.num_qubits == 2)
    return dict(depth=tqc.depth(), depth_2q=d2, n_2q=n2q,
                n_1q=sum(v for g, v in ops.items() if g in ("rz", "sx", "x")),
                n_ops=sum(v for g, v in ops.items() if g not in ("barrier", "measure", "delay")),
                phys_qubits=len(active))

# ---------- statistics ----------
def wilson(s, n, z=1.959964):
    p = s / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def _nll(params, ks, succ, shots, N, model):
    if model == "1p":
        lam, A = params[0], 1.0
    else:
        lam, A = params
    q = 1 - lam
    pk = A * q ** ks * p_ideal(ks, N) + (1 - A * q ** ks) / N
    pk = np.clip(pk, 1e-12, 1 - 1e-12)
    return -np.sum(succ * np.log(pk) + (shots - succ) * np.log(1 - pk))

def fit_lambda(ks, succ, shots, N, model="1p"):
    """Binomial MLE of lambda_eff (model '1p': Eq. 18) or (lambda, A) (model '2p':
    A <= 1 absorbs a k-independent SPAM/initialisation loss)."""
    from scipy.optimize import minimize, minimize_scalar
    ks, succ, shots = map(np.asarray, (ks, succ, shots))
    if model == "1p":
        r = minimize_scalar(lambda l: _nll([l], ks, succ, shots, N, "1p"),
                            bounds=(0.0, 0.999), method="bounded")
        return dict(lam=r.x, A=1.0, nll=r.fun)
    best = None
    for l0 in (0.05, 0.3, 0.7):
        r = minimize(lambda p: _nll(p, ks, succ, shots, N, "2p"), x0=[l0, 0.9],
                     bounds=[(0.0, 0.999), (0.0, 1.0)], method="L-BFGS-B")
        if best is None or r.fun < best.fun:
            best = r
    return dict(lam=best.x[0], A=best.x[1], nll=best.fun)

def expected_cost(ks, phat):
    phat = np.maximum(np.asarray(phat, float), 1e-12)
    return (np.asarray(ks) + 1) / phat

def speedup_from_cost(N, cost):
    return (N + 1) / 2 / np.min(cost)


## Connect to IBM Quantum (or the fake backend in test mode)

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
service = None
if MODE == "test":
    from qiskit_ibm_runtime.fake_provider import FakeFez
    backend = FakeFez()
else:
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient(); token = _s.get_secret("IBM_QUANTUM_TOKEN")
        try:
            instance = _s.get_secret("IBM_QUANTUM_INSTANCE")
        except Exception:
            instance = None
    except Exception:
        token = os.environ.get("IBM_QUANTUM_TOKEN"); instance = os.environ.get("IBM_QUANTUM_INSTANCE")
    assert token, "IBM_QUANTUM_TOKEN secret not found (Kaggle: Add-ons -> Secrets)"
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token, instance=instance or None)
    if MODE == "retrieve":
        assert JOB_ID or os.path.exists(f"{OUT}/job_id.json"), "set JOB_ID"
        JOB_ID = JOB_ID or json.load(open(f"{OUT}/job_id.json"))["job_id"]
        backend = service.job(JOB_ID).backend()
    elif BACKEND_NAME:
        backend = service.backend(BACKEND_NAME)
    else:
        backend = service.least_busy(operational=True, simulator=False, min_num_qubits=10)
print("backend:", backend.name, "| qubits:", backend.num_qubits)

## Build and transpile the circuits; save resources and the calibration snapshot

In [ ]:
def target_of(n): return 107 % 2 ** n
from qiskit import qpy
if MODE == "retrieve" and os.path.exists(f"{OUT}/circuits.qpy"):
    # use exactly the circuits that were submitted (saved at submission)
    with open(f"{OUT}/circuits.qpy", "rb") as fh: circs = qpy.load(fh)
    meta = pd.read_csv(f"{OUT}/hw_resources.csv")
else:
    circs, meta = [], []
    for st in SEEDS_TRANSPILER:
        for n in NS:
            for k in range(2 * r_opt(2 ** n) + 3):
                tq = transpile(grover_circuit(n, target_of(n), k, "v-chain"), backend, optimization_level=3,
                               seed_transpiler=int(st))
                circs.append(tq); meta.append(dict(seed=int(st), n=n, k=k, **resources(tq),
                                                   layout=str(tq.layout.final_index_layout()) if tq.layout else ""))
    meta = pd.DataFrame(meta); meta.to_csv(f"{OUT}/hw_resources.csv", index=False)
    with open(f"{OUT}/circuits.qpy", "wb") as fh: qpy.dump(circs, fh)
print(len(circs), "circuits; total shots:", len(circs) * SHOTS)
print(meta.groupby(["seed", "n"])[["depth", "depth_2q", "n_2q", "phys_qubits"]].max())

def gate_errors(be, tq):
    lf = 0.0
    for inst in tq.data:
        nm = inst.operation.name
        if nm in ("measure", "barrier", "delay"): continue
        qa = tuple(tq.find_bit(q).index for q in inst.qubits)
        try: e = be.target[nm][qa].error or 0.0
        except Exception: e = 0.0
        lf += np.log1p(-min(e, 0.999999))
    return lf
snap = dict(backend=backend.name, utc=datetime.datetime.utcnow().isoformat(), mode=MODE, run_tag=RUN_TAG,
            seeds_transpiler=SEEDS_TRANSPILER, shots=SHOTS)
json.dump(snap, open(f"{OUT}/hw_snapshot.json", "w"), indent=1)

## Run (test: Aer with the fake backend's noise; submit: real QPU; retrieve: load an earlier job)

In [ ]:
from qiskit_aer import AerSimulator
def counts_from_sampler_result(res):
    out = []
    for i, pub in enumerate(res):
        out.append(pub.data.c.get_counts())
    return out
if MODE == "test":
    raw = AerSimulator.from_backend(backend).run(circs, shots=SHOTS, seed_simulator=7).result()
    hw_counts = [raw.get_counts(i) for i in range(len(circs))]
else:
    if MODE == "submit":
        sampler = SamplerV2(mode=backend)
        sampler.options.default_shots = SHOTS
        job = sampler.run(circs)
        JOB_ID = job.job_id()
        json.dump(dict(job_id=JOB_ID, backend=backend.name, submitted_utc=datetime.datetime.utcnow().isoformat()),
                  open(f"{OUT}/job_id.json", "w"), indent=1)
        print("submitted job", JOB_ID, "- if the session ends, use MODE='retrieve' with this JOB_ID")
    job = service.job(JOB_ID)
    print("status:", job.status())
    res = job.result()                     # waits for completion
    hw_counts = counts_from_sampler_result(res)
    try:
        print("QPU usage (s):", job.usage_estimation, "|", job.metrics().get("usage", {}))
    except Exception as ex:
        print("usage info not available:", ex)
rows = []
for (_, r), cnt in zip(meta.iterrows(), hw_counts):
    key = format(target_of(int(r.n)), f"0{int(r.n)}b")
    rows.append(dict(seed=int(r.seed), n=int(r.n), k=int(r.k), shots=sum(cnt.values()), succ=cnt.get(key, 0)))
hw = pd.DataFrame(rows); hw.to_csv(f"{OUT}/hw_counts.csv", index=False); hw.head()

## Noisy simulation of the same transpiled circuits with the device's calibration (no QPU time)

In [ ]:
sim = AerSimulator.from_backend(backend)
sr = []
for s in range(3):
    raw = sim.run(circs, shots=SHOTS, seed_simulator=100 + s).result()
    for i, (_, r) in enumerate(meta.iterrows()):
        key = format(target_of(int(r.n)), f"0{int(r.n)}b")
        sr.append(dict(layout_seed=int(r.seed), n=int(r.n), k=int(r.k), seed=s, shots=SHOTS, succ=raw.get_counts(i).get(key, 0)))
simc = pd.DataFrame(sr); simc.to_csv(f"{OUT}/hw_sim_counts.csv", index=False)

## Analysis: λ_eff (hardware vs simulation vs calibration), cost-optimal k, speedup
Uncertainty: parametric bootstrap of the shot counts, pooled over the transpiler seeds (layouts). Per-layout results are in `hw_summary_by_layout.csv`.

In [ ]:
from scipy.stats import chi2
def analyse(df, N):
    a = df.groupby("k")[["succ", "shots"]].sum(); ks = a.index.values
    f1, f2 = fit_lambda(ks, a.succ.values, a.shots.values, N, "1p"), fit_lambda(ks, a.succ.values, a.shots.values, N, "2p")
    ph = a.succ.values / a.shots.values; c = (ks + 1) / np.maximum(ph, 1e-12)
    rng = np.random.default_rng(N); bs = []
    for _ in range(2000):
        pb = rng.binomial(a.shots.values, ph) / a.shots.values
        bs.append((N + 1) / 2 / ((ks + 1) / np.maximum(pb, 1e-12)).min())
    kg = np.arange(0, 4 * int(np.sqrt(N)) + 21)
    k1 = int(kg[np.argmin((kg + 1) / p_noisy(kg, N, f1["lam"]))])
    return dict(lam_1p=f1["lam"], lam_2p=f2["lam"], A_2p=f2["A"], k_emp=int(ks[np.argmin(c)]), k_1p=k1,
                regret_1p=c[min(k1, len(c) - 1)] / c.min() - 1, S_hat=(N + 1) / 2 / c.min(),
                S_lo=np.percentile(bs, 2.5), S_hi=np.percentile(bs, 97.5),
                S_1p=(N + 1) / 2 / ((kg + 1) / p_noisy(kg, N, f1["lam"])).min())
summ = []
for n in NS:
    N = 2 ** n; K = 2 * r_opt(N) + 2
    s0 = SEEDS_TRANSPILER[0]
    lf0 = gate_errors(backend, circs[[i for i, (_, r) in enumerate(meta.iterrows()) if r.n == n and r.k == 0 and r.seed == s0][0]])
    lfK = gate_errors(backend, circs[[i for i, (_, r) in enumerate(meta.iterrows()) if r.n == n and r.k == K and r.seed == s0][0]])
    lam_cal = 1 - np.exp((lfK - lf0) / K)
    for src, d in [("hardware", hw[hw.n == n]), ("simulation", simc[simc.n == n])]:
        summ.append(dict(source=src, n=n, N=N, R_opt=r_opt(N), lam_cal=lam_cal, **analyse(d, N)))
summ = pd.DataFrame(summ); summ.to_csv(f"{OUT}/hw_summary.csv", index=False)
byl = []
for n in NS:
    for st in SEEDS_TRANSPILER:
        byl.append(dict(n=n, seed=st, **analyse(hw[(hw.n == n) & (hw.seed == st)], 2 ** n)))
pd.DataFrame(byl).to_csv(f"{OUT}/hw_summary_by_layout.csv", index=False)
print(backend.name, "|", snap["utc"]); summ.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(NS), figsize=(3.2 * len(NS), 2.8))
for ax, n in zip(axes, NS):
    N = 2 ** n
    for src, d, m in [("hardware", hw[hw.n == n].groupby("k")[["succ", "shots"]].sum().reset_index(), "o"), ("simulation", simc[simc.n == n].groupby("k")[["succ", "shots"]].sum().reset_index(), "s")]:
        p = d.succ / d.shots; lo, hi = wilson(d.succ.values, d.shots.values)
        ax.errorbar(d.k, p, yerr=[p - lo, hi - p], fmt=m, ms=3, label=src, capsize=1)
    kk = np.arange(0, 2 * r_opt(N) + 3); lam = summ[(summ.source == "hardware") & (summ.n == n)].lam_1p.item()
    ax.plot(kk, p_ideal(kk, N), "k--", lw=1, label="ideal"); ax.plot(kk, p_noisy(kk, N, lam), "-", lw=1, label="Eq. 18 (hw fit)")
    ax.axhline(1 / N, color="grey", ls=":", lw=0.6); ax.set_title(f"{backend.name}, n={n}", fontsize=8); ax.set_xlabel("k")
axes[0].set_ylabel("P(success)"); axes[0].legend(fontsize=6); fig.tight_layout(); fig.savefig(f"{OUT}/fig_hw_success_vs_k.png", dpi=250)

## What to send back
Download the whole `results_hw_<RUN_TAG>/` folder (Output tab) and send it. It contains:
- `hw_counts.csv` and `hw_sim_counts.csv`;
- `hw_resources.csv`, `hw_summary.csv` and `hw_snapshot.json`;
- `hw_summary_by_layout.csv`, `circuits.qpy`;
- `job_id.json` and the figure.

These are enough to add a hardware subsection to the paper.